# 02 - 提示模板系统

## 学习目标

1. 使用 `ChatPromptTemplate.from_messages()` 构建对话提示
2. 掌握 `SystemMessage`、`HumanMessage`、`AIMessage` 的语义和用法
3. 使用 `MessagesPlaceholder` 插入对话历史
4. 模板变量：`{variable}` 语法和部分填充
5. 构建 `FewShotChatMessagePromptTemplate`：示例选择器 + 示例提示
6. 使用 `SemanticSimilarityExampleSelector` 进行动态示例选择
7. 组合多个模板：`ChatPromptTemplate.from_messages` 混合模式
8. 完整管道：`template | model | StrOutputParser()`

In [ ]:
# 基础导入
import sys
sys.path.insert(0, '../..')

from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
    AIMessagePromptTemplate,
    MessagesPlaceholder,
    FewShotChatMessagePromptTemplate,
    PromptTemplate,
)
from langchain_core.prompts import FewShotPromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# Mock LLM（复用01课的模型，这里重建一个独立的）
from langchain_core.language_models import BaseChatModel
from langchain_core.messages import BaseMessage
from langchain_core.outputs import ChatResult, ChatGeneration
from typing import Any


class MockChatModel(BaseChatModel):
    """Mock ChatModel 用于演示提示模板。
    
    回显接收到的消息结构，帮助理解提示模板的输出格式。
    """
    
    def _generate(
        self,
        messages: list[BaseMessage],
        stop: list[str] | None = None,
        run_manager: Any = None,
        **kwargs: Any,
    ) -> ChatResult:
        # 回显消息结构
        message_summary = []
        for i, msg in enumerate(messages):
            msg_type = type(msg).__name__
            content = msg.content if hasattr(msg, 'content') else str(msg)
            preview = content[:60] + "..." if len(content) > 60 else content
            message_summary.append(f"  [{i}] {msg_type}: {preview}")
        
        response = "收到以下消息:\n" + "\n".join(message_summary)
        message = AIMessage(content=response)
        return ChatResult(generations=[ChatGeneration(message=message)])
    
    @property
    def _llm_type(self) -> str:
        return "mock-chat-model"


mock_model = MockChatModel()
print("MockChatModel 已就绪。")

---

## 1. ChatPromptTemplate 基础

### 概念说明

`ChatPromptTemplate` 是 LangChain 中构建聊天模型提示的核心类。
与纯文本 `PromptTemplate` 不同，它生成的是**消息列表**，每条消息都有一个**角色**：

- **SystemMessage**：系统指令，设定 AI 的行为和角色
- **HumanMessage**：用户消息，代表用户的输入
- **AIMessage**：AI 消息，代表模型的回复

`from_messages()` 方法接收一个消息模板列表，支持两种简洁格式：
- `("system", "text")` 元组格式
- `MessagePromptTemplate` 对象格式

### 示例 1.1：最简单的 ChatPromptTemplate

In [ ]:
# 使用元组格式创建提示模板（推荐方式）
simple_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个友好的AI助手，名叫小智。请用中文回答。"),
    ("human", "{user_input}"),
])

# 查看模板的消息结构
print("=== 模板消息结构 ===")
for msg in simple_prompt.messages:
    print(f"  类型: {type(msg).__name__}")
    print(f"  内容: {msg.prompt.template if hasattr(msg, 'prompt') else msg}")
    print()

# 格式化模板
formatted_messages = simple_prompt.format_messages(user_input="今天天气怎么样？")
print("=== 格式化后的消息列表 ===")
for i, msg in enumerate(formatted_messages):
    print(f"[{i}] {type(msg).__name__}: {msg.content}")

### 示例 1.2：三种消息类型的显式创建

In [ ]:
# 使用 MessagePromptTemplate 显式创建
explicit_prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(
        "你是一个{role}，擅长{skill}。请始终用{language}回答。"
    ),
    HumanMessagePromptTemplate.from_template("{question}"),
])

# 获取输入变量
print(f"模板需要的输入变量: {explicit_prompt.input_variables}")

# 格式化
messages = explicit_prompt.format_messages(
    role="数学老师",
    skill="代数教学",
    language="中文",
    question="请解释一元二次方程的求根公式。",
)

print("\n=== 格式化消息 ===")
for i, msg in enumerate(messages):
    print(f"[{i}] {type(msg).__name__}:")
    print(f"    {msg.content}")
    print()

### 示例 1.3：包含 AI 消息的模板（多轮对话预设）

可以在模板中预设 AI 的回复，用于示例或角色设定。

In [ ]:
# 包含预设 AI 回复的模板
conversation_preset = ChatPromptTemplate.from_messages([
    ("system", "你是一个客服助手。"),
    ("human", "你好，我需要帮助。"),
    ("ai", "您好！很高兴为您服务，请问有什么可以帮您的？"),
    ("human", "{user_query}"),
])

# 格式化
messages = conversation_preset.format_messages(user_query="我想查询我的订单状态。")

print("=== 预设对话模板格式化结果 ===")
for i, msg in enumerate(messages):
    role = type(msg).__name__.replace("Message", "")
    print(f"[{i}] {role}: {msg.content[:80]}")

---

## 2. MessagesPlaceholder：对话历史插入

### 概念说明

`MessagesPlaceholder` 是对话式 AI 应用的关键组件。
它在模板中预留一个位置，用于插入**可变数量的历史消息**。

典型用途：
```python
ChatPromptTemplate.from_messages([
    ("system", "..."),
    MessagesPlaceholder(variable_name="history"),  # 历史消息插入点
    ("human", "{question}"),
])
```

调用时，`history` 参数接收一个消息列表，会被展开插入到占位符位置。

### 示例 2.1：基础历史插入

In [ ]:
# 创建带历史占位符的模板
history_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个智能助手，请根据对话历史回答问题。"),
    MessagesPlaceholder(variable_name="chat_history"),  # 历史插入点
    ("human", "{user_input}"),
])

# 模拟对话历史
chat_history = [
    HumanMessage(content="你好，我叫张三。"),
    AIMessage(content="你好张三！很高兴认识你。"),
    HumanMessage(content="我住在北京。"),
    AIMessage(content="北京是个很棒的城市！"),
]

# 格式化模板
messages = history_prompt.format_messages(
    chat_history=chat_history,
    user_input="我叫什么名字？我住在哪里？"
)

print("=== 带历史的提示模板结果 ===")
for i, msg in enumerate(messages):
    role = type(msg).__name__.replace("Message", "")
    content = msg.content if hasattr(msg, 'content') else str(msg)
    print(f"[{i}] {role}: {content[:80]}")

print(f"\n总消息数: {len(messages)}")
print(f"其中历史消息数: {len(chat_history)}")
print(f"系统消息 + 占位符展开 + 用户输入 = 1 + {len(chat_history)} + 1 = {len(messages)}")

### 示例 2.2：可选历史（optional=True）

当 `optional=True` 时，即使不传入历史消息也不会报错。

In [ ]:
# 可选历史占位符
optional_history_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个智能助手。"),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{user_input}"),
])

# 场景1：有历史
with_history = optional_history_prompt.format_messages(
    chat_history=[HumanMessage(content="Hi"), AIMessage(content="Hello")],
    user_input="How are you?"
)
print(f"有历史: {len(with_history)} 条消息")

# 场景2：无历史（不会报错）
without_history = optional_history_prompt.format_messages(
    user_input="How are you?"
)
print(f"无历史: {len(without_history)} 条消息")

# 场景3：空历史列表
empty_history = optional_history_prompt.format_messages(
    chat_history=[],
    user_input="How are you?"
)
print(f"空历史: {len(empty_history)} 条消息")

### 示例 2.3：集成到 LCEL 链中

展示历史模板与管道操作符的结合使用。

In [ ]:
# 构建带历史支持的完整链
history_aware_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个知识渊博的助手，请基于对话历史回答问题。"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}"),
])

# 使用 MockChatModel
history_chain = history_aware_prompt | mock_model | StrOutputParser()

# 模拟一轮对话
history = []

def chat_turn(user_input: str) -> str:
    """模拟一轮对话，自动更新历史。"""
    global history
    response = history_chain.invoke({
        "history": history,
        "question": user_input,
    })
    # 更新历史
    history.append(HumanMessage(content=user_input))
    history.append(AIMessage(content=response))
    return response


# 模拟多轮对话
print("=== 多轮对话模拟 ===")
questions = [
    "什么是向量数据库？",
    "它和传统数据库有什么区别？",
    "能举个具体的例子吗？",
]

for i, q in enumerate(questions, 1):
    response = chat_turn(q)
    print(f"\n第{i}轮:")
    print(f"  用户: {q}")
    print(f"  助手: {response[:80]}...")

print(f"\n历史记录中共有 {len(history)} 条消息 ({len(history)//2} 轮对话)")

---

## 3. 模板变量与部分填充

### 概念说明

模板变量使用 `{variable_name}` 语法嵌入到提示文本中。
LangChain 提供了两种填充方式：

1. **完全填充**：调用 `.invoke()` 或 `.format_messages()` 时提供所有变量
2. **部分填充**：使用 `.partial()` 方法预先填充部分变量，返回一个新的模板

`.partial()` 在以下场景非常有用：
- **分阶段构建**：在不同阶段填充不同变量
- **固定默认值**：将某些变量设为常量
- **动态函数值**：使用函数动态生成变量值

### 示例 3.1：.partial() 基础用法

In [ ]:
# 创建一个需要多个变量的模板
multi_var_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是{company}公司的{role}，你的风格是{style}。"),
    ("human", "{question}"),
])

print(f"原始模板变量: {multi_var_prompt.input_variables}")

# 部分填充：固定公司和角色
partial_prompt = multi_var_prompt.partial(
    company="AICorp",
    role="技术支持工程师",
)

print(f"部分填充后变量: {partial_prompt.input_variables}")

# 最后填充剩余变量
messages = partial_prompt.format_messages(
    style="专业而友好",
    question="软件安装失败怎么办？"
)

print("\n=== 分步填充结果 ===")
for i, msg in enumerate(messages):
    print(f"[{i}] {type(msg).__name__}: {msg.content}")

### 示例 3.2：使用函数进行部分填充

`.partial()` 可以接收函数作为值，该函数会在每次调用时动态生成内容。

In [ ]:
from datetime import datetime

# 创建需要当前日期的模板
dated_prompt = ChatPromptTemplate.from_messages([
    ("system", "当前日期是{current_date}。你是{assistant_name}。"),
    ("human", "{question}"),
])

# 使用函数动态填充日期
dated_prompt_partial = dated_prompt.partial(
    current_date=lambda: datetime.now().strftime("%Y年%m月%d日 %H:%M"),
    assistant_name="小智",
)

# 多次调用，日期会动态变化
print("=== 动态日期填充 ===")
for i in range(3):
    messages = dated_prompt_partial.format_messages(question="今天有什么新闻？")
    # 提取 system 消息中的日期
    sys_msg = messages[0].content
    print(f"第{i+1}次调用: {sys_msg[:60]}...")
    import time as _time
    _time.sleep(1)  # 等1秒看时间变化

### 示例 3.3：在 LCEL 链中使用 partial

部分填充模板在构建可重用链时非常有用。

In [ ]:
# 创建一个通用的系统提示模板
base_system_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是{role}。\n\n核心原则：\n{principles}"),
    ("human", "{question}"),
])

# 通过 partial 创建不同角色的链
teacher_chain = (
    base_system_prompt.partial(
        role="一位耐心的编程老师",
        principles="1. 用简单易懂的语言解释\n2. 提供代码示例\n3. 鼓励学生动手实践"
    )
    | mock_model
    | StrOutputParser()
)

reviewer_chain = (
    base_system_prompt.partial(
        role="一位严格的代码审查员",
        principles="1. 检查代码规范\n2. 发现潜在Bug\n3. 提出优化建议"
    )
    | mock_model
    | StrOutputParser()
)

# 测试
question = "这段代码有什么问题？ x = [1,2,3]; for i in range(len(x)): print(x[i])"

teacher_response = teacher_chain.invoke({"question": question})
reviewer_response = reviewer_chain.invoke({"question": question})

print("=== 老师视角 ===")
print(teacher_response[:100] + "...")
print("\n=== 审查员视角 ===")
print(reviewer_response[:100] + "...")

---

## 4. FewShotChatMessagePromptTemplate

### 概念说明

Few-shot prompting（少样本提示）是在提示中提供示例来引导模型行为的技术。

`FewShotChatMessagePromptTemplate` 由两部分组成：

1. **示例选择器（Example Selector）**：从示例库中选出最相关的示例
2. **示例格式化器（Example Prompt）**：将选中的示例格式化为消息

```python
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_selector=selector,     # 根据输入选择示例
    example_prompt=example_prompt, # 如何格式化每个示例
)
```

### 示例 4.1：静态示例选择器（固定示例）

当示例数量较少且固定时，使用 `from_messages` 直接内嵌示例。

In [ ]:
# 准备示例数据
examples = [
    {
        "input": "将'你好'翻译成英文",
        "output": "Hello"
    },
    {
        "input": "将'谢谢'翻译成英文",
        "output": "Thank you"
    },
    {
        "input": "将'再见'翻译成英文",
        "output": "Goodbye"
    },
]

# 创建示例格式化模板
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("ai", "{output}"),
])

# 使用 LangChain 内置的基于长度的示例选择器
# 这里先展示手动构建 FewShotChatMessagePromptTemplate 的方式
from langchain_core.prompts.few_shot import FewShotChatMessagePromptTemplate

# 方式1：直接传入 examples（不使用选择器）
few_shot_prompt = FewShotChatMessagePromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
)

# 查看 few-shot 提示生成的示例消息
example_messages = few_shot_prompt.format_messages()
print(f"Few-shot 示例消息数: {len(example_messages)}")
for i, msg in enumerate(example_messages):
    print(f"  [{i}] {type(msg).__name__}: {msg.content}")

### 示例 4.2：FewShotChatMessagePromptTemplate 完整链

将 few-shot 示例嵌入到完整的聊天模板中，并构建 LLM 链。

In [ ]:
# 构建包含 few-shot 示例的完整提示
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个中英翻译助手。请参考以下示例进行翻译，只返回翻译结果。"),
    few_shot_prompt,  # 内嵌 few-shot 示例
    ("human", "{input}"),
])

# 构建翻译链
translation_chain = final_prompt | mock_model | StrOutputParser()

# 测试
test_inputs = [
    "将'早上好'翻译成英文",
    "将'晚安'翻译成英文",
]

print("=== 少样本翻译测试 ===")
for inp in test_inputs:
    result = translation_chain.invoke({"input": inp})
    print(f"输入: {inp}")
    print(f"输出: {result[:50]}...")
    print()

---

## 5. SemanticSimilarityExampleSelector

### 概念说明

当示例库很大时，不应该把所有示例都塞进提示（会超出上下文窗口）。
`SemanticSimilarityExampleSelector` 通过**语义相似度**从示例库中动态选择与当前输入最相关的 top-k 个示例。

工作原理：
1. 将所有示例的输入文本向量化并存入向量存储
2. 收到新输入时，进行相似度搜索
3. 返回最相似的 k 个示例

这需要嵌入模型和向量存储的支持。为了演示，我们使用 Mock 嵌入。

### 示例 5.1：使用 SemanticSimilarityExampleSelector

In [ ]:
from langchain_core.example_selectors import SemanticSimilarityExampleSelector
from langchain_community.vectorstores import Chroma

# Mock Embeddings 类（用于演示语义选择器，无需 API 调用）
from langchain_core.embeddings import Embeddings


class MockEmbeddings(Embeddings):
    """简单的 Mock 嵌入，使用字符哈希生成伪向量。
    
    仅用于演示语义选择器的工作流程。
    生产环境中请使用 OpenAIEmbeddings 或其他真实嵌入模型。
    """
    
    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return [self._hash_embed(text) for text in texts]
    
    def embed_query(self, text: str) -> list[float]:
        return self._hash_embed(text)
    
    def _hash_embed(self, text: str) -> list[float]:
        """生成一个简单的伪嵌入向量（64维）。
        
        使用字符的 Unicode 码点生成。相同的文本产生相同的向量。
        """
        import hashlib
        # 使用 SHA256 哈希生成确定性的种子
        hash_bytes = hashlib.sha256(text.encode('utf-8')).digest()
        # 将哈希字节转换为 64 维浮点数向量
        vector = []
        for i in range(0, min(len(hash_bytes), 64)):
            # 将字节值映射到 [-1, 1] 范围
            vector.append((hash_bytes[i] / 255.0) * 2 - 1)
        # 补齐到 64 维
        while len(vector) < 64:
            vector.append(0.0)
        return vector


# 准备更大的示例库
translation_examples = [
    {"input": "将'你好'翻译成英文", "output": "Hello"},
    {"input": "将'谢谢'翻译成英文", "output": "Thank you"},
    {"input": "将'再见'翻译成英文", "output": "Goodbye"},
    {"input": "将'苹果'翻译成英文", "output": "Apple"},
    {"input": "将'香蕉'翻译成英文", "output": "Banana"},
    {"input": "将'猫'翻译成英文", "output": "Cat"},
    {"input": "将'狗'翻译成英文", "output": "Dog"},
    {"input": "将'书'翻译成英文", "output": "Book"},
    {"input": "将'水'翻译成英文", "output": "Water"},
    {"input": "将'太阳'翻译成英文", "output": "Sun"},
]

# 创建语义相似度选择器
example_selector = SemanticSimilarityExampleSelector.from_examples(
    examples=translation_examples,
    embeddings=MockEmbeddings(),
    vectorstore_cls=Chroma,
    k=3,  # 每次选择最相似的3个示例
    input_keys=["input"],  # 指定用于计算相似度的字段
)

print(f"示例库大小: {len(translation_examples)}")
print(f"每次返回: k=3 个示例")

# 测试：根据输入选择示例
test_query = "将'橘子'翻译成英文"
selected = example_selector.select_examples({"input": test_query})
print(f"\n查询: '{test_query}'")
print(f"选中的 {len(selected)} 个最相似示例:")
for i, ex in enumerate(selected):
    print(f"  [{i}] {ex['input']} → {ex['output']}")

### 示例 5.2：将选择器集成到 FewShotChatMessagePromptTemplate

In [ ]:
# 创建基于选择器的 FewShotChatMessagePromptTemplate
dynamic_few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_selector=example_selector,
    example_prompt=ChatPromptTemplate.from_messages([
        ("human", "{input}"),
        ("ai", "{output}"),
    ]),
    input_variables=["input"],  # 需要传给选择器的变量
)

# 构建完整的动态 few-shot 提示
dynamic_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个翻译助手。参考以下相似示例进行翻译，只输出翻译结果。"),
    dynamic_few_shot_prompt,
    ("human", "{input}"),
])

# 构建链
dynamic_chain = dynamic_prompt | mock_model | StrOutputParser()

# 测试不同的查询，验证选择器选择了不同的示例
test_queries = [
    "将'橙子'翻译成英文",
    "将'老师'翻译成英文",
    "将'月亮'翻译成英文",
]

print("=== 动态示例选择测试 ===")
for query in test_queries:
    selected = example_selector.select_examples({"input": query})
    selected_inputs = [ex['input'] for ex in selected]
    result = dynamic_chain.invoke({"input": query})
    print(f"\n查询: {query}")
    print(f"  选中的示例输入: {selected_inputs}")
    print(f"  链输出: {result[:60]}...")

---

## 6. 提示组合：混合使用多种消息类型

### 概念说明

`ChatPromptTemplate.from_messages()` 的强大之处在于它支持**混合使用**各种消息类型：

```python
ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate,       # 系统指令
    MessagesPlaceholder,               # 对话历史
    FewShotChatMessagePromptTemplate,  # 少样本示例
    HumanMessagePromptTemplate,        # 用户输入
])
```

这种组合能力使你可以构建复杂但清晰的多层次提示结构。

### 示例 6.1：全功能组合提示

构建一个包含所有组件的复合提示：系统指令 + 少样本示例 + 对话历史 + 用户输入。

In [ ]:
# 全功能复合提示
composite_prompt = ChatPromptTemplate.from_messages([
    # 1. 系统指令（使用 SystemMessagePromptTemplate）
    SystemMessagePromptTemplate.from_template(
        "你是{assistant_name}，一个{domain}领域的专家。回答要求：{requirements}"
    ),
    
    # 2. Few-shot 示例
    FewShotChatMessagePromptTemplate(
        examples=[
            {"input": "Python是什么？", "output": "Python是一种高级编程语言，以其简洁的语法和丰富的库生态而闻名。"},
            {"input": "什么是REST API？", "output": "REST API是一种基于HTTP协议的应用程序接口设计风格，使用标准的HTTP方法进行CRUD操作。"},
        ],
        example_prompt=ChatPromptTemplate.from_messages([
            ("human", "{input}"),
            ("ai", "{output}"),
        ]),
    ),
    
    # 3. 对话历史占位符（可选）
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    
    # 4. 用户输入
    HumanMessagePromptTemplate.from_template("{user_question}"),
])

# 格式化测试
messages = composite_prompt.format_messages(
    assistant_name="TechBot",
    domain="计算机科学",
    requirements="简洁准确，附带代码示例",
    chat_history=[
        HumanMessage(content="你好"),
        AIMessage(content="你好！有什么技术问题我可以帮你？"),
    ],
    user_question="什么是Docker？",
)

print("=== 复合提示结构 ===")
print(f"总消息数: {len(messages)}")
for i, msg in enumerate(messages):
    role = type(msg).__name__.replace("Message", "")
    content = msg.content[:80] if hasattr(msg, 'content') else str(msg)[:80]
    print(f"[{i}] {role:6s}: {content}...")

print(f"\n结构分析:")
print(f"  1条 SystemMessage (系统指令)")
print(f"  4条来自 FewShot (2对 human+ai)")
print(f"  2条来自 chat_history")
print(f"  1条 HumanMessage (用户问题)")
print(f"  总计: {1 + 4 + 2 + 1} = {len(messages)}")

### 示例 6.2：分层构建策略

在实际开发中，推荐使用**分层构建**策略：先构建基础模板，再逐步添加功能层。

In [ ]:
def build_qa_prompt(
    system_template: str,
    include_history: bool = False,
    include_examples: bool = False,
    examples: list[dict] | None = None,
) -> ChatPromptTemplate:
    """分层构建 QA 提示模板。
    
    Args:
        system_template: 系统提示文本
        include_history: 是否包含对话历史占位符
        include_examples: 是否包含少样本示例
        examples: 示例列表（include_examples=True 时需要）
    
    Returns:
        构建完成的 ChatPromptTemplate
    """
    message_templates = []
    
    # 第1层：系统指令（始终存在）
    message_templates.append(("system", system_template))
    
    # 第2层：少样本示例（可选）
    if include_examples and examples:
        message_templates.append(
            FewShotChatMessagePromptTemplate(
                examples=examples,
                example_prompt=ChatPromptTemplate.from_messages([
                    ("human", "{input}"),
                    ("ai", "{output}"),
                ]),
            )
        )
    
    # 第3层：对话历史（可选）
    if include_history:
        message_templates.append(MessagesPlaceholder(variable_name="history"))
    
    # 第4层：用户输入（始终存在）
    message_templates.append(("human", "{question}"))
    
    return ChatPromptTemplate.from_messages(message_templates)


# 测试不同配置
print("=== 分层构建测试 ===\n")

# 配置 A：仅系统指令
prompt_a = build_qa_prompt("你是一个AI助手。")
print(f"配置A（基础）:")
print(f"  变量: {prompt_a.input_variables}")
print(f"  消息数: {len(prompt_a.messages)}")

# 配置 B：系统 + 历史
prompt_b = build_qa_prompt("你是一个AI助手。", include_history=True)
print(f"\n配置B（+历史）:")
print(f"  变量: {prompt_b.input_variables}")
print(f"  消息数: {len(prompt_b.messages)}")

# 配置 C：系统 + 示例
prompt_c = build_qa_prompt(
    "你是一个AI助手。",
    include_examples=True,
    examples=[{"input": "Q1", "output": "A1"}],
)
print(f"\n配置C（+示例）:")
print(f"  变量: {prompt_c.input_variables}")
print(f"  消息数: {len(prompt_c.messages)}")

# 配置 D：全部
prompt_d = build_qa_prompt(
    "你是一个AI助手。",
    include_history=True,
    include_examples=True,
    examples=[{"input": "Q1", "output": "A1"}],
)
print(f"\n配置D（全部）:")
print(f"  变量: {prompt_d.input_variables}")
print(f"  消息数: {len(prompt_d.messages)}")

---

## 7. 完整管道：template | model | StrOutputParser()

### 概念说明

现在将所有知识整合起来，构建各种类型的完整 LLM 管道。

```
输入字典 → ChatPromptTemplate → ChatModel → StrOutputParser → 字符串输出
```

### 示例 7.1：多角色管道

根据输入参数动态选择不同的系统提示，构建条件化管道。

In [ ]:
from langchain_core.runnables import RunnableBranch

# 定义不同角色的提示
coder_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个Python高级开发者。请提供代码和解释。"),
    ("human", "{question}"),
])

teacher_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个编程入门老师，用最简单的方式解释。"),
    ("human", "{question}"),
])

default_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个通用AI助手。"),
    ("human", "{question}"),
])


# 定义路由条件
def needs_coding_expertise(input_data: dict) -> bool:
    question = input_data.get("question", "")
    code_keywords = ["代码", "编程", "debug", "Python", "算法", "函数", "类"]
    return any(kw in question for kw in code_keywords)

def needs_simple_explanation(input_data: dict) -> bool:
    question = input_data.get("question", "")
    simple_keywords = ["什么是", "解释", "入门", "基础", "初学"]
    return any(kw in question for kw in simple_keywords)


# 构建路由管道
routed_chain = RunnableBranch(
    (needs_coding_expertise, coder_prompt),
    (needs_simple_explanation, teacher_prompt),
    default_prompt,
) | mock_model | StrOutputParser()


# 测试
test_questions = [
    "这段Python代码有什么问题？",
    "什么是机器学习？",
    "今天天气怎么样？",
]

print("=== 多角色路由管道 ===")
for q in test_questions:
    result = routed_chain.invoke({"question": q})
    print(f"\n问题: {q}")
    print(f"回答: {result[:80]}...")

### 示例 7.2：带验证的完整生产管道

构建一个包含输入验证、模板处理、模型调用、输出解析的完整管道。

In [ ]:
class InputValidationError(Exception):
    """输入验证错误。"""
    pass


def validate_qa_input(input_data: dict) -> dict:
    """验证 QA 输入。"""
    if not isinstance(input_data, dict):
        raise InputValidationError(f"输入必须是字典，收到 {type(input_data)}")
    
    question = input_data.get("question", "")
    if not question or not str(question).strip():
        raise InputValidationError("question 字段不能为空")
    
    if len(str(question).strip()) < 3:
        raise InputValidationError(f"问题至少需要3个字符，当前:{len(question)}")
    
    # 标准化
    return {
        **input_data,
        "question": str(question).strip(),
        "validated_at": datetime.now().isoformat(),
    }


def truncate_output(text: str, max_length: int = 500) -> str:
    """截断过长的输出。"""
    if len(text) > max_length:
        return text[:max_length] + "...\n\n[回答过长，已截断]"
    return text


# 构建生产级管道
production_qa_prompt = ChatPromptTemplate.from_messages([
    ("system", """你是{assistant_name}，一个专业的问答助手。

回答规则：
{rules}

当前时间：{current_time}"""),
    MessagesPlaceholder(variable_name="history", optional=True),
    ("human", "{question}"),
])

# 使用 partial 固定部分变量
production_qa_prompt = production_qa_prompt.partial(
    assistant_name="知识小助手",
    rules="""1. 回答要准确、简洁
2. 如果不知道，请诚实说明
3. 使用中文回答""",
    current_time=lambda: datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
)

# 组装完整管道
production_chain = (
    RunnableLambda(validate_qa_input)  # 步骤1: 验证输入
    | production_qa_prompt              # 步骤2: 构建提示
    | mock_model                        # 步骤3: 模型推理
    | StrOutputParser()                 # 步骤4: 解析输出
    | RunnableLambda(truncate_output)   # 步骤5: 截断处理
)

# 测试正常输入
print("=== 正常输入测试 ===")
try:
    valid_result = production_chain.invoke({
        "question": "什么是RAG？",
        "history": [],
    })
    print(f"成功: {valid_result[:80]}...")
except Exception as e:
    print(f"失败: {e}")

# 测试异常输入
print("\n=== 异常输入测试 ===")
invalid_inputs = [
    {},  # 缺少 question
    {"question": ""},  # 空问题
    {"question": "ab"},  # 太短
]
for invalid in invalid_inputs:
    try:
        production_chain.invoke(invalid)
    except InputValidationError as e:
        print(f"正确捕获验证错误: {e}")
    except Exception as e:
        print(f"其他错误: {type(e).__name__}: {e}")

---

## 8. 总结与最佳实践

### 提示模板设计原则

1. **分层构建**：将系统指令、示例、历史、用户输入分层组织
2. **使用 partial**：固定不变的部分，减少变量数量
3. **语义选择器**：大规模示例库必须使用选择器，避免超出上下文窗口
4. **验证输入**：在管道开始处添加验证步骤
5. **MessagesPlaceholder**：始终设为 `optional=True`，提高健壮性

### 模板类型选择指南

| 场景 | 推荐模板 |
|------|----------|
| 简单问答 | `ChatPromptTemplate.from_messages([("system", ...), ("human", ...)])` |
| 多轮对话 | + `MessagesPlaceholder` |
| 行为引导 | + `FewShotChatMessagePromptTemplate` + 静态示例 |
| 大规模示例 | + `SemanticSimilarityExampleSelector` |
| 多角色 | + `RunnableBranch` 路由不同模板 |
| 分阶段填充 | + `.partial()` 逐步固定变量 |

### 自我检测

1. `MessagesPlaceholder` 与普通模板变量有什么区别？
2. `.partial()` 在管道的哪个阶段调用最合适？
3. `SemanticSimilarityExampleSelector` 的 `k` 参数应该如何选择？
4. 如何验证模板的所有变量都被正确填充？

In [ ]:
# 综合练习：构建一个完整的多语言翻译助手

def build_translation_assistant(
    source_lang: str,
    target_lang: str,
    style: str = "formal",
) -> "RunnableSequence":
    """构建多语言翻译助手。
    
    Args:
        source_lang: 源语言
        target_lang: 目标语言
        style: 翻译风格 (formal/casual/literal)
    
    Returns:
        LCEL 翻译链
    """
    style_instructions = {
        "formal": "请使用正式、规范的表达方式。",
        "casual": "请使用日常口语化的表达方式。",
        "literal": "请进行逐字逐句的直译。",
    }
    
    system_text = (
        f"你是一个{source_lang}到{target_lang}的专业翻译。\n"
        f"{style_instructions.get(style, style_instructions['formal'])}\n"
        "只输出翻译结果，不要添加任何解释。"
    )
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_text),
        ("human", "请翻译以下文本：\n{text}"),
    ])
    
    return prompt | mock_model | StrOutputParser()


# 测试不同配置
print("=== 多语言翻译助手测试 ===\n")

# 中译英（正式）
zh_en_formal = build_translation_assistant("中文", "英文", "formal")
result = zh_en_formal.invoke({"text": "人工智能正在改变世界。"})
print(f"中→英 (正式): {result[:60]}...")

# 中译英（口语）
zh_en_casual = build_translation_assistant("中文", "英文", "casual")
result = zh_en_casual.invoke({"text": "你今天过得怎么样？"})
print(f"中→英 (口语): {result[:60]}...")

# 测试不支持的风格（应回退到默认）
zh_en_default = build_translation_assistant("中文", "英文", "poetic")
result = zh_en_default.invoke({"text": "落霞与孤鹜齐飞。"})
print(f"中→英 (默认): {result[:60]}...")

---

## 本课小结

在本课中，你学习了 LangChain 提示模板系统的完整知识：

1. **ChatPromptTemplate**：`from_messages()` 构建对话提示
2. **消息类型**：SystemMessage、HumanMessage、AIMessage 的语义和用法
3. **MessagesPlaceholder**：插入对话历史，支持 `optional` 模式
4. **模板变量**：`{variable}` 语法和 `.partial()` 分阶段填充
5. **FewShotChatMessagePromptTemplate**：静态示例和选择器模式
6. **SemanticSimilarityExampleSelector**：基于语义相似度的动态示例选择
7. **提示组合**：分层构建全功能复合提示
8. **完整管道**：`template | model | parser` 的生产级实现

这些提示模板技术是构建高质量 RAG 应用的基石。